# 02 — Field-conditioned synthetic geology and exact AVO generation

| Item | Definition |
|---|---|
| **Scientific purpose** | Turn the calibrated Stage-01 structural/elastic background into geologically diverse, physics-generated AVO realizations. |
| **Inputs** | Stage-01 Vp, Vs, density, DELTA/P(sand), porosity, RGT, reservoir mask, stratigraphic fraction, blend weights, and the fitted reservoir elastic model. |
| **Outputs** | Complete realization packages containing geology, brine and substituted elastic properties, dense-angle exact PP AVO, three angle stacks, PWD dip, masks, coordinates, and provenance. |
| **Data availability** | Algorithms and configuration schemas are public. Licensed S01 inputs and generated arrays remain local. |
| **Local data requirements** | `work_data_root` and `private_artifact_root` are defined in ignored `configs/paths.yaml`. Execution stops if the required Stage-01 artifacts are unavailable. |
| **Software requirements** | `pip install -e ".[field,ml,notebooks]"`; Madagascar is optional and used only for independent reference-path cross-checking. |
| **Approximate runtime** | Minutes per realization on CPU; the configured 100-realization production family is an offline generation job. |
| **Pipeline position** | Consumes Notebook 01; produces the full realizations consumed by Notebook 03. |

The main forward operator is the **exact PP Zoeppritz solution** followed by wavelet convolution and angle-domain mute/taper. Shuey/Aki–Richards intercept and gradient are compact diagnostics and later model features—not substitutes for the generation physics.

In [ ]:
from pathlib import Path

def find_repository_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "sage_avo").exists():
            return candidate
    raise RuntimeError("SAGE-AVO repository root not found; start the kernel within an installed checkout.")

ROOT = find_repository_root()

import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sage_avo.config import load_config, seed_everything
from sage_avo.experiments import (
    generate_stage02_dataset,
    load_stage01_background,
    load_stage02_manifest,
)
from sage_avo.forward import (
    ForwardConfig,
    forward_avo_dense,
    forward_avo_madagascar,
    forward_specification_from_mapping,
    madagascar_availability,
)

paths_file = ROOT / "configs" / "paths.yaml"
if not paths_file.exists():
    raise FileNotFoundError(
        "Missing local configuration: create configs/paths.yaml from "
        "configs/paths.example.yaml and define the authorized input and artifact roots."
    )
paths = load_config(paths_file)
private_root = Path(paths["private_artifact_root"])
validation_root_text = os.getenv("SAGE_AVO_REVISION3_VALIDATION_ROOT", "").strip()
if validation_root_text:
    validation_root = Path(validation_root_text)
    workflow = json.loads((validation_root / "configs" / "synthetic_resolved.json").read_text())
    realization_dir = validation_root / "stage02" / "realizations"
    figure_dir = validation_root / "figures" / "stage02"
else:
    workflow = load_config(ROOT / "configs" / "synthetic_s01_v0032.yaml")
    support_contract = load_config(ROOT / "configs" / "revision331_support_acceptance.yaml")
    workflow["stage"].update({
        "name": "field_conditioned_synthetic_avo_v00331_support_aware",
        "geology_realization_count": 100,
        "observation_variants_per_geology": 1,
        "realization_count": 100,
        "realization_id_offset": 3_400_000,
        "member_master_seeds": list(range(3_400_000, 3_400_100)),
    })
    workflow["fluid_substitution"].update({
        "enabled": True,
        "mode": "calibrated_differential_gassmann",
        "calibration_id": "v0033_58a5fe39a11c4fe66431",
        "calibration_artifact": "derived/fluid_models_v0033/calibrated_dry_frame_scenario_ensemble.npz",
        "fluid_property_validation_artifact": "derived/fluid_models_v0033/fluid_property_validation.json",
    })
    workflow["support_aware_acceptance"] = support_contract
    workflow["outputs"].update({
        "version": "v00331_production100_support_aware",
        "directory": "synthetic/v00331_production100_support_aware/realizations",
    })
    realization_dir = private_root / "stage_artifacts" / "stage02" / workflow["outputs"]["version"] / "realizations"
    figure_dir = private_root / "figures" / "revision331" / "stage02_production"
seed_everything(int(workflow["stage"]["seed"]))
figure_dir.mkdir(parents=True, exist_ok=True)

## 1. Stage-01 contract and conventions

All channels share the Stage-01 time/CDP grid. `elastic_background` and `elastic_blend_weight` have shape `[3, time, trace]`; the other image channels have shape `[time, trace]`.

The canonical convention is

\[
\mathrm{DELTA}=\text{shaliness},\qquad P(\mathrm{sand})=1-\mathrm{DELTA}.
\]

The interface stores both channels, enforces `DELTA + P(sand) = 1`, and records the convention in every realization manifest. The configured 0.30 sand threshold follows the calibrated Stage-01 reservoir probability distribution; it is not a generic 0.5 classifier threshold.

In [ ]:
stage01, reservoir_model, source_hashes = load_stage01_background(
    paths["work_data_root"],
    workflow["inputs"]["dataset_id"],
    workflow["inputs"]["structure_version"],
)
contract = pd.DataFrame(
    [{"channel": name, "shape": value.shape, "dtype": value.dtype} for name, value in stage01.items()]
)
display(contract)
print(f"Hashed source artifacts: {len(source_hashes)}")
print(
    "Reservoir P(sand) range:",
    np.nanmin(stage01["sand_probability"][stage01["reservoir_mask"].astype(bool)]),
    np.nanmax(stage01["sand_probability"][stage01["reservoir_mask"].astype(bool)]),
)

## 2. Deterministic geological realization

A geology-realization ID is its geological random seed. One coherent deformation field is applied to every Stage-01 channel, so horizons, RGT, facies, porosity, masks, and elastic background remain registered. The deformation combines smooth folds with optional finite-length fault displacement. Correlated Gaussian fields perturb P(sand), porosity, saturation, and coupled bulk/shear/density properties before forward modeling. Observation-variant IDs use separate post-forward seeds; every variant of one geology is kept in one ML split.

The trained Stage-01 random-forest relationship maps `[DELTA, porosity, stratigraphic fraction]` to reservoir Vp/Vs/density. Warped regional elastic background is preserved outside the reservoir and blended only across the saved transition weights; this prevents the block artifacts produced by assigning a constant exterior.

In [ ]:
print("Configured realizations:", workflow["stage"]["realization_count"])
print("Geological deformation parameters:", {
    key: value for key, value in workflow["geology"].items()
    if "fold" in key or "fault" in key
})
print("Sand facies P(sand) threshold:", workflow["geology"]["sand_facies_probability_threshold"])
print("Fluid substitution:", workflow["fluid_substitution"])

## 3. CO₂ scenario and fluid substitution

CO₂ saturation is introduced only in connected, sufficiently thick reservoir sand. Fluid substitution is selected explicitly by the versioned configuration and recorded in each realization manifest. Production-eligible modes preserve a common dry frame and transfer the fluid-induced bulk-modulus and density response while keeping shear modulus invariant; they also require a validated pressure/temperature/fluid-property artifact. Compatibility modes that overwrite or project the empirical random-forest brine state are excluded from production claims. Both brine and substituted elastic cubes are retained for physical QC.

## 4. Exact dense-angle forward response

For each elastic interface and each configured angle, the shared production specification solves the exact isotropic PP Zoeppritz system, retaining the real component of the complex post-critical solution. The configured wavelet bank is convolved with explicit constant-zero same-length boundaries, then the global-time front mute/taper is applied. Dense responses retain all 43 angles from 3° through 45°. The identical serialized specification is consumed by the differentiable Stage-04 operator.

Production angle bands are centralized in configuration with declared shared endpoints: near `3–17°`, mid `17–31°`, and far `31–45°`. The overlap at 17° and 31° is intentional and hashed into the forward contract. Compact P/G representative angles are the band midpoints (`10°`, `24°`, `38°`) rather than an independent forward convention.

In [ ]:
forward_definition = forward_specification_from_mapping(workflow)
display(
    pd.DataFrame(
        [{"band": b.name, "minimum_deg": b.minimum_degrees, "maximum_deg": b.maximum_degrees}
         for b in forward_definition.bands]
    )
)
print("Forward specification SHA-256:", forward_definition.sha256)
print("Production bands:", workflow["forward_model"]["bands"])
print("Post-forward observation perturbations:", workflow["observation_perturbations"])
print("Dense angles:", forward_definition.angles_degrees)

## 5. Generate the realization family

The default call creates the complete configured family. `SAGE_AVO_STAGE02_LIMIT` creates an explicitly labeled `operator_validation_subset`; corpus-level analysis accepts only a manifest labeled as complete. `SAGE_AVO_REUSE_STAGE02=1` reopens an existing immutable artifact set.

In [ ]:
limit_text = os.getenv("SAGE_AVO_STAGE02_LIMIT", "").strip()
realization_limit = int(limit_text) if limit_text else None
reuse = os.getenv("SAGE_AVO_REUSE_STAGE02", "0") == "1"
workers = int(os.getenv("SAGE_AVO_STAGE02_WORKERS", "1"))
manifest_path = realization_dir / "manifest.json"

if reuse and manifest_path.exists():
    manifest = load_stage02_manifest(manifest_path)
else:
    manifest = generate_stage02_dataset(
        config=workflow,
        paths=paths,
        output_directory=realization_dir,
        realization_limit=realization_limit,
        workers=workers,
        resume=reuse,
    )

display(pd.Series({key: manifest[key] for key in (
    "status", "requested_realizations", "generated_realizations", "exact_forward_operator"
)}).to_frame("value"))

## 6. Deterministic realization QC

The representative realization is the smallest generated ID—a documented rule independent of visual appearance. The panels verify channel registration, the DELTA/P(sand) complement, plume support, corrected local fluid substitution, exact near/mid/far response, and recalculated PWD dip. RGT is coherently warped from Stage 01; dip is recalculated for structural QC rather than replacing the warped RGT graph coordinate.

In [ ]:
representative_id = min(manifest["realization_ids"])
representative_path = realization_dir / f"realization_{representative_id:07d}.npz"
with np.load(representative_path, allow_pickle=False) as archive:
    realization = {name: archive[name] for name in archive.files}

panels = [
    (realization["sand_probability"], "P(sand)", "viridis"),
    (realization["delta"], "DELTA (shaliness)", "viridis_r"),
    (realization["porosity"], "Porosity", "viridis"),
    (realization["co2_saturation"], "CO₂ saturation", "magma"),
    (realization["elastic"][0], "Vp", "viridis"),
    (realization["elastic"][1], "Vs", "viridis"),
    (realization["elastic"][2], "Density", "viridis"),
    (realization["rgt"], "Warped RGT", "turbo"),
    (realization["avo"][0], "Near AVO", "gray"),
    (realization["avo"][1], "Mid AVO", "gray"),
    (realization["avo"][2], "Far AVO", "gray"),
    (realization["dip_pwd"], "Recalculated PWD dip", "coolwarm"),
]
fig, axes = plt.subplots(3, 4, figsize=(16, 10), constrained_layout=True)
for axis, (array, title, cmap) in zip(axes.flat, panels):
    image = axis.imshow(array, aspect="auto", cmap=cmap)
    axis.set_title(title)
    axis.set_xlabel("Trace")
    axis.set_ylabel("Time sample")
    fig.colorbar(image, ax=axis, shrink=0.72)
for axis in axes[:2].flat:
    top_sample = np.interp(realization["horizon_top_ms"], realization["time_ms"], np.arange(realization["time_ms"].size))
    base_sample = np.interp(realization["horizon_base_ms"], realization["time_ms"], np.arange(realization["time_ms"].size))
    axis.plot(top_sample, color="white", linewidth=0.8, label="warped T6")
    axis.plot(base_sample, color="black", linewidth=0.8, label="warped T7")
fig.suptitle(f"Stage-02 field-conditioned realization {representative_id}")
qc_path = figure_dir / "stage02_representative_realization.png"
fig.savefig(qc_path, dpi=300, bbox_inches="tight")
plt.show()

print("max |DELTA + P(sand) - 1| =", np.max(np.abs(
    realization["delta"] + realization["sand_probability"] - 1.0
)))
print("Saved figure:", qc_path)

## 7. Mandatory Stage-02/Stage-04 operator round trip

The saved clean three-band AVO is regenerated from the saved truth elastic model with the differentiable Torch operator and the same hashed forward specification. This checks exact PP reflectivity—including post-critical complex slowness—wavelet convolution, global mute/taper, and inclusive band stacking. Reported tolerances are numerical, not visually judged.

In [ ]:
import torch
from sage_avo.forward import forward_avo_three_band_spec_torch

elastic64 = torch.from_numpy(realization["elastic"].astype(np.float64))
reproduced = forward_avo_three_band_spec_torch(
    elastic64[0][None], elastic64[1][None], elastic64[2][None], forward_definition
)[0].detach().numpy()
difference = reproduced - realization["avo_clean"]
round_trip = {
    "maximum_absolute_error": float(np.max(np.abs(difference))),
    "rmse": float(np.sqrt(np.mean(difference**2))),
    "relative_rmse": float(
        np.sqrt(np.mean(difference**2))
        / max(np.sqrt(np.mean(realization["avo_clean"] ** 2)), 1e-15)
    ),
}
display(pd.Series(round_trip).to_frame("value"))

## 8. Madagascar reference-path cross-check

If Madagascar is installed, the same elastic crop is passed through `sfzoeppritz2 → sftransp → sfricker1 → sftransp`. Correlation is the principal diagnostic because `sfricker1` and the NumPy implementation use different wavelet normalization conventions. This reference check does not change the backend recorded in the realization manifest.

In [ ]:
availability = madagascar_availability()
print(availability)
if availability.available:
    crop = realization["elastic"][:, 80:130, 20:120]
    controlled_wavelet = forward_definition.wavelets[0]
    madagascar_definition = ForwardConfig(
        angles_degrees=forward_definition.angles_degrees,
        bands=forward_definition.bands,
        wavelet_hz=controlled_wavelet.peak_frequency_hz,
        dt_seconds=forward_definition.dt_seconds,
        wavelet_samples=controlled_wavelet.samples,
        apply_mute=forward_definition.apply_mute,
        mute_start=forward_definition.mute_start,
        mute_end=forward_definition.mute_end,
        taper_samples=forward_definition.taper_samples,
    )
    numpy_forward = forward_avo_dense(*crop, config=madagascar_definition)
    rsf_forward = forward_avo_madagascar(*crop, config=madagascar_definition)
    comparisons = []
    for band, numpy_stack, rsf_stack in zip(
        numpy_forward.band_names, numpy_forward.stacks, rsf_forward.stacks
    ):
        comparisons.append({
            "band": band,
            "correlation": np.corrcoef(numpy_stack.ravel(), rsf_stack.ravel())[0, 1],
            "standard_deviation_ratio_rsf_to_numpy": rsf_stack.std() / numpy_stack.std(),
        })
    display(pd.DataFrame(comparisons))
else:
    print("Madagascar cross-check skipped; exact NumPy Zoeppritz remains the configured operator.")

## 9. Saved-channel manifest

In [ ]:
channel_table = pd.DataFrame(
    [{"channel": name, **definition} for name, definition in manifest["channels"].items()]
)
display(channel_table)

## Stage outputs

| artifact | shape/type | scientific meaning | consumed by |
|---|---|---|---|
| `realization_XXXXXXX.npz` | dense 43-angle AVO; 3-band AVO; 3-channel elastic; geological/structural masks | One deterministic field-conditioned geological and exact-physics experiment | Notebook 03 |
| per-realization JSON | provenance + deformation/fluid parameters + QC | Reproducibility record | Notebooks 03 and 05 |
| `manifest.json` | channel schema, hashes, IDs, conventions | Immutable Stage-02 dataset contract | Notebook 03 |

## Scientific checks

- Shared deformation keeps geology, RGT, masks, and elastic fields registered.
- `DELTA + P(sand) = 1` is checked numerically.
- Elastic and AVO channels are finite and physically bounded in each saved QC record.
- Zero saturation reproduces the local RF brine state; outside-plume values remain unchanged; dry-frame shear is retained.
- Corrected CO₂ substitution is confined to connected reservoir sand; brine and substituted elastic cubes are both retained.
- Exact dense-angle Zoeppritz is the primary operator; compact P/G approximations are not used to generate training observations.
- Wavelet, convolution, mute, angle bands, and post-forward perturbations are persisted per realization.
- Observation variants share an explicit geology split-group ID.
- Optional Madagascar correlation checks the independent reference route.

## Next stage

Notebook 03 consumes the immutable realization IDs and complete saved channels. It splits at the **realization level**, constructs the disclosed truth-derived low-frequency elastic prior, and extracts traceable multiscale patches without leakage.